# E1-open on Colab

Validate the Prefixwise classifier without API credits, using an open Hugging Face model on vLLM, with controlled injected causes as ground truth.

**Before running:** Runtime > Change runtime type > GPU (T4 or L4). Default model is Qwen3-4B (Colab default, fits on T4/L4). Qwen3-8B failed to start on a Colab L4 in the Nov 5-6, 2026 live run ("RuntimeError: Engine core initialization failed", server never ready in 10 min); 8B needs a larger GPU (RTX 4090 / A100 class). If you have one, set `MODEL = "Qwen/Qwen3-8B"` below.

No API keys needed. Qwen3-4B is not gated, so no Hugging Face token is needed.


In [ ]:
# Config: Colab default is Qwen3-4B (8B failed to start on a Colab L4, Nov 5-6 2026).
# On a larger GPU (RTX 4090 / A100), set MODEL = "Qwen/Qwen3-8B".
MODEL = "Qwen/Qwen3-4B"
MODEL_ALIAS = MODEL + "-alias"
VLLM_PORT = 8000

import subprocess, sys
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)


In [ ]:
# Install vLLM (>= 0.30.0, for session_id on KV events) and Prefixwise.
# This takes a few minutes on first run.
!pip install -q 'vllm>=0.30.0' pytest
!git clone https://github.com/tejachavali/prefixwise.git /tmp/prefixwise || (cd /tmp/prefixwise && git pull)
!cd /tmp/prefixwise && git log --oneline -1

In [ ]:
# Start the vLLM server in the background (same flags as vllm_server.sh).
import subprocess, time, urllib.request, json, os

log = open("/tmp/vllm.log", "w")
proc = subprocess.Popen(
    ["vllm", "serve", MODEL,
     "--host", "127.0.0.1", "--port", str(VLLM_PORT),
     "--served-model-name", MODEL, MODEL_ALIAS,
     "--enable-prefix-caching", "--enable-prompt-tokens-details",
     # vLLM 0.33.0 rejects tool-carrying requests with HTTP 400
     # ("`auto` tool choice requires --enable-auto-tool-choice and
     # --tool-call-parser") unless these are set. Qwen3 emits
     # Hermes-style tool calls, so the hermes parser is used.
     "--enable-auto-tool-choice", "--tool-call-parser", "hermes"],
    stdout=log, stderr=subprocess.STDOUT, cwd="/tmp/prefixwise")
print(f"vLLM starting, pid={proc.pid}, log=/tmp/vllm.log")

# Wait for the server to be ready (model load can take several minutes).
for i in range(60):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{VLLM_PORT}/health", timeout=5) as r:
            print("vLLM ready:", r.read()[:200])
            break
    except Exception as e:
        time.sleep(10)
        if i % 6 == 0:
            print(f"waiting... {i*10}s, last error: {e}")
else:
    print("vLLM did not become ready. Check /tmp/vllm.log")
    print(open("/tmp/vllm.log").read()[-4000:])

In [ ]:
# Run E1-open: 100 turns, injected causes as ground truth.
!cd /tmp/prefixwise && PYTHONPATH=src python3 experiments/e1_open/e1_open.py --vllm-url http://127.0.0.1:8000 --model $MODEL --model-alias $MODEL_ALIAS

In [ ]:
# Show the score report (newest file in results/).
import glob
files = sorted(glob.glob("/tmp/prefixwise/experiments/e1_open/results/*.md"))
print(files[-1] if files else "no report yet")
if files:
    print(open(files[-1]).read())

## Interpreting the result

- **Pass:** agreement >= 95%, UNKNOWN <= 10% of misses.
- **Mock vs live:** this notebook is the live run, so a pass here is real evidence about vLLM and the classifier. The `--mock` mode in `e1_open.py` is only a harness smoke test.
- **If MODEL_SWITCH fails:** the alias may not have been served; the report marks those turns `upstream_error=True`. Check `/tmp/vllm.log` for the served-model-name error.
- **If RETRY_REPLAY fails:** retry turns use a fresh upstream session id and a fresh `cache_salt` per turn, so the vLLM prefix is cold (cached_tokens == 0). The old `/reset_prefix_cache` endpoint is no longer used: it returns HTTP 404 on vLLM 0.33.0 (live run Nov 5-6, 2026, retry_replay 0/10). If a retry turn still scores as NONE, check its `cached_tokens` in the report: a high value means the prefix was not actually cold on that vLLM version.
- **Qwen3-8B on Colab L4:** failed to start in the Nov 5-6, 2026 live run ("RuntimeError: Engine core initialization failed"). Use Qwen3-4B on Colab; 8B needs a larger GPU.
